# Hockey Teams: Forms, Searching and Pagination
## Build a web scraper that can conduct searches and paginate through the results.

In [ ]:
pip install pyarrow

In [ ]:
pip install fastparquet

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
import os 
import time

In [10]:

headers = {"User-Agent": "Mozilla/5.0"}


def safe_int(tag):
    return int(tag.text.strip()) if tag and tag.text.strip() else None


def safe_float(tag):
    return float(tag.text.strip()) if tag and tag.text.strip() else None


def parse_html(soup):
    """
    Parses BeautifulSoup object and extracts team data.
    """
    rows = soup.select("tr.team")
    data = []

    for row in rows:
        data.append({
            "Name": row.select_one("td.name").text.strip(),
            "Year": safe_int(row.select_one("td.year")),
            "Wins": safe_int(row.select_one("td.wins")),
            "Losses": safe_int(row.select_one("td.losses")),
            "Win %": safe_float(row.select_one("td.pct")),
            "Goals For": safe_int(row.select_one("td.gf")),
            "Goals Against": safe_int(row.select_one("td.ga")),
            "+/-": safe_int(row.select_one("td.diff"))
        })

    return data


def get_total_pages(soup):
    """
    Extracts total number of pages from pagination section.
    """
    pages = soup.select("ul.pagination li a")
    return max([int(p.text) for p in pages if p.text.strip().isdigit()], default=1)


def fetch_and_parse(query=None):
    """
    Fetches hockey team data with pagination.
    Optionally filters by query.
    Saves raw HTML and structured dataset.
    """
    folder = "hockey_teams_saved_html"
    os.makedirs(folder, exist_ok=True)

    base_url = "https://www.scrapethissite.com/pages/forms/"

    base_params = {"page_num": 1}
    if query:
        base_params["q"] = query

    # First request to detect total pages
    res = requests.get(base_url, headers=headers, params=base_params, timeout=10)
    res.raise_for_status()

    soup = BeautifulSoup(res.text, "lxml")
    last_page = get_total_pages(soup)

    print(f"Total pages: {last_page}")

    all_data = []

    for page_num in range(1, last_page + 1):
        page_params = base_params.copy()
        page_params["page_num"] = page_num

        try:
            res = requests.get(base_url, headers=headers, params=page_params, timeout=10)
            res.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"Error on page {page_num}: {e}")
            break

        soup = BeautifulSoup(res.text, "lxml")
        rows = soup.select("tr.team")

        if not rows:
            print(f"No data on page {page_num}, stopping early.")
            break

        # Parse once using soup
        page_data = parse_html(soup)
        all_data.extend(page_data)

        # Save raw HTML
        file_name = (
            f"{folder}/page_{page_num}.html"
            if not query
            else f"{folder}/{query}_page_{page_num}.html"
        )

        with open(file_name, "w", encoding="utf-8") as f:
            f.write(res.text)

        print(f"Saved: {file_name}")

        time.sleep(1)

    # Convert to DataFrame
    df = pd.DataFrame(all_data)

    print(f"\nTotal records collected: {len(df)}")
    print(df.head())

    # Save dataset
    output_file = "clean_dataset.csv" if not query else f"{query}_dataset.csv"
    df.to_csv(output_file, index=False)
    df.to_parquet("hockey_teams.parquet", index=False)

    print(f"\nDataset saved as: {output_file}")

    return df


# Example usage
# df = fetch_and_parse()
df = fetch_and_parse("Boston")

Total pages: 1
Saved: hockey_teams_saved_html/Boston_page_1.html

Total records collected: 21
            Name  Year  Wins  Losses  Win %  Goals For  Goals Against  +/-
0  Boston Bruins  1990    44      24  0.550        299            264   35
1  Boston Bruins  1991    36      32  0.450        270            275   -5
2  Boston Bruins  1992    51      26  0.607        332            268   64
3  Boston Bruins  1993    42      29  0.500        289            252   37
4  Boston Bruins  1994    27      18  0.562        150            127   23

Dataset saved as: Boston_dataset.csv
